In [ ]:
from src.dataset_processing import create_dataloaders
from src.train_loop import train_model, model_num_params, train_model_with_RL
from src.models.ConvLSTM import ConvLSTM
from src.models.ConvSelfAttention import ConvSelfAttention
from src.models.AdoptiveDepth import DeepShallowRL, QLearningAgent, get_initial_state, translate_action_to_dynamic_layer_counts
from src.models.BiDepth import BidirectionalDeepShallow

In [ ]:
import h5py    
import numpy as np    

file_name = "../data/weather/train_test_2015_2020.h5"

with h5py.File(file_name, "r") as f:
    print("Keys: %s" % f.keys())

    data_dict = {}
    for k, v in f.items():
        data_dict[k] = np.array(v)
        


In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt


def show_weather_data(data_Precipitation ,data_Latitude , data_Longitude, SHOW_US_MAP = False):
    
    # Creating a new DataFrame with the flattened data
    df_flat = pd.DataFrame({
        'Latitude': np.array(data_Latitude).flatten(),
        'Longitude': np.array(data_Longitude).flatten(),
        'Precipitation': data_Precipitation.flatten()
    })

    if SHOW_US_MAP:
        # # Convert the DataFrame to a GeoDataFrame
        gdf_flat = gpd.GeoDataFrame(df_flat, geometry=gpd.points_from_xy(df_flat.Longitude, df_flat.Latitude))
        
        # Load a map of the US for background
        world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
        usa = world[world.name == 'United States of America']
        
        
        # Plotting again
        fig, ax = plt.subplots(1, 1, figsize=(15, 10))
        
        # Plot heat map
        gdf_flat.plot(ax=ax, column='Precipitation', cmap='viridis', legend=True, markersize=10)
        
        # Set plot limits to the extent of your data
        ax.set_xlim(df_flat['Longitude'].min(), df_flat['Longitude'].max())
        ax.set_ylim(df_flat['Latitude'].min(), df_flat['Latitude'].max())
        
        # Replot USA map for boundaries on top
        usa.plot(ax=ax, color='none', edgecolor='white')
        
        # Setting labels and title
        ax.set_title('Precipitation Levels Across the Continental United States')
        ax.set_xlabel('Longitude')
        ax.set_ylabel('Latitude')

    else:
        # Attempting the heat map visualization again
        plt.figure(figsize=(10, 6))
        plt.scatter(df_flat['Longitude'], df_flat["Latitude"], c=df_flat['Precipitation'], cmap='viridis')
        plt.colorbar(label='Precipitation (mm)')
        plt.xlabel('Longitude')
        plt.ylabel('Latitude')
        plt.title('Heat Map of Precipitation by Latitude and Longitude')

    plt.show()

In [ ]:
len( data_dict['Precipitation'])/(24*4)/365

In [ ]:
show_weather_data( data_dict['Precipitation'][1010],  data_dict['Latitude'] , data_dict['Longitude'] , SHOW_US_MAP = True)

In [ ]:
train_loader, val_loader, test_loader = create_dataloaders(
    data_dict['Precipitation'], 
    frame_number=8, 
    window_size=6, 
    batch_size=32, 
    train_split=0.85, 
    val_split=0.05
)

In [ ]:
len(train_loader)

In [ ]:
for data in test_loader:
    break

In [ ]:
data[0].shape

In [ ]:
data[0].shape

In [ ]:
import torch

input_dim = 8
kernel_size = (3,3)
num_layers = 3
hidden_dim=[16, 16,  8]
in_channels = 8
out_channels = 8
window = 6

initial_layers = 3
attention = True


# conv_lstm = ConvLSTM(input_dim, hidden_dim, kernel_size, num_layers, in_channels, out_channels, data[0].shape[-2:], window)
# conv_lstm = ConvSelfAttention(input_dim, hidden_dim, kernel_size, num_layers, in_channels, out_channels, data[0].shape[-2:], window)
conv_lstm = BidirectionalDeepShallow(input_dim, hidden_dim, kernel_size, num_layers, in_channels, out_channels,  data[0].shape[-2:] ,window, initial_layers = initial_layers, attention=attention)

input1 = torch.randn(data[0].shape)

conv_lstm(input1).shape
# BimodelConv(input1).shape

In [ ]:

# Example Setup
window_size = 6
# initial_loss = 10
state_size = 100
max_depth= 5

# Initialize the QLearningAgent
agent = QLearningAgent(window_size, 3.2)  # Example state and action sizes

# Get the initial state
init_action , current_layers = get_initial_state(3, window_size)

# Choose an action and translate it to dynamic layer counts
action = agent.choose_action(3)
# action = action - window_size  # Subtract 7 to get actions in the range [-7, 6]

new_layer_counts = translate_action_to_dynamic_layer_counts(action , current_layers.copy(), max_depth = max_depth, window_size =window_size )

# Print the results
print(f"Initial layer counts: {current_layers}")
print(f"Chosen action: {action}")
print(f"New layer counts: {new_layer_counts}")



In [ ]:
import torch

input_dim = 8
kernel_size = (3,3)
num_layers = 3
hidden_dim=[16, 16,  8]
in_channels = 8
out_channels = 8
window = 6
initial_layers = 10
dynamic_layer_counts = [5, 4, 10, 3, 1, 10]

model = DeepShallowRL(input_dim, hidden_dim, kernel_size, num_layers, in_channels, out_channels, data[0].shape[-2:] ,window, initial_layers = initial_layers , attention = False)

input1 = torch.randn(32, 6, 8, 512, 256) # (batch , days (window)  , min_int (channels) , x , y )
# input2 = torch.randn(2 , 16, 27, 8) # (batch , days (window)  , min_int (channels) , x , y )

model.update_dynamic_layer_counts(dynamic_layer_counts)

model(input1).shape

In [ ]:
model.load_state_dict(torch.load("checkpoint_RL.pt"))

In [ ]:
data[1].shape

In [ ]:
data_output_0 = model(data[0])

In [ ]:
data_dict['Precipitation'][0].shape

In [ ]:
data_output_0[0][0]

In [ ]:
data[1][0][0]

In [ ]:
show_weather_data( data_output_0[0][0].detach().numpy(),  data_dict['Latitude'] , data_dict['Longitude'] , SHOW_US_MAP = True)

In [ ]:
show_weather_data( data[1][0][0],  data_dict['Latitude'] , data_dict['Longitude'] , SHOW_US_MAP = True)

In [ ]:
import torch.nn as nn

criterion = nn.MSELoss()

criterion(data[1], data_output_0)

In [ ]:
# Is MPS even available? macOS 12.3+
print(torch.backends.mps.is_available())

# Was the current version of PyTorch built with MPS activated?
print(torch.backends.mps.is_built())


In [ ]:
model.update_dynamic_layer_counts([1,1,1,1,1,1])

model_num_params(model)


In [ ]:
import torch.optim as optim
from tqdm.autonotebook import tqdm
import torch.optim.lr_scheduler as lr_scheduler
import logging
import torch.nn as nn
import torch



# Setup logging
logging.basicConfig(filename='training_log.log', level=logging.INFO, format='%(asctime)s %(message)s')




input_dim = 8
kernel_size = (3,3)
num_layers = 3
hidden_dim=[16, 16,  8]
in_channels = 8
out_channels = 8
window_size = 6
initial_layers = 10
max_depth = 10

q_learning_agent = QLearningAgent(window_size, 5)


if torch.backends.mps.is_available():
    device = torch.device("mps")
    print( 'Mac MPS is avilable' )
else:
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    if torch.cuda.is_available() : 
        print( 'Cuda is avilable' )

# Create an instance of the CNNRNN
# model = ConvLSTM(input_dim, hidden_dim, kernel_size, num_layers, in_channels, out_channels, data[0].shape[-2:], window).to(device)
# model = ConvSelfAttention(input_dim, hidden_dim, kernel_size, num_layers, in_channels, out_channels, tensor_do.shape[-2:]  , window ).to(device)
# model = BidirectionalDeepShallow(input_dim, hidden_dim, kernel_size, num_layers, in_channels, out_channels,  data[0].shape[-2:] ,window, initial_layers = initial_layers, attention=attention).to(device)
model = DeepShallowRL(input_dim, hidden_dim, kernel_size, num_layers, in_channels, out_channels, data[0].shape[-2:] ,window_size, initial_layers = initial_layers , attention = False).to(device)


# Define the loss function and optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters())

# Define the learning rate decay scheduler
# scheduler = lr_scheduler.ExponentialLR(optimizer, gamma=0.99)


# Define the number of training epochs
scheduler = None
num_epochs = 10
# validation_interval = 5
validation_interval = 100 # Now Refers to batch count
patience = 10



# model, train_losses, val_losses, _  = train_model(
#     model,
#     criterion,
#     optimizer,
#     train_loader,
#     val_loader,
#     num_epochs,
#     validation_interval,
#     patience,
#     scheduler=None,
#     print_losses=True,
# )

model, train_losses, val_losses, _  = train_model_with_RL(
    model,
    criterion,
    optimizer,
    train_loader,
    val_loader,
    test_loader,
    num_epochs,
    validation_interval,
    patience,
    q_learning_agent,  # Instance of QLearningAgent
    window_size,
    max_depth,
    logging,
    device,
    scheduler=None,
    print_losses=True,
)